# M5 Forecasting Accuracy — EDA
**Walmart の商品別日次販売数を予測する時系列コンペ**

- 30,490 商品 × 1,969 日 (2011-01-29 〜 2016-06-19)
- 3州 (CA, TX, WI) × 10店舗 × 3カテゴリ (HOBBIES, HOUSEHOLD, FOODS)
- 評価指標: **WRMSSE** (Weighted Root Mean Squared Scaled Error)
- 予測対象: 最後28日間 (d_1914〜d_1941)

### データ構成
| ファイル | 行数 | 内容 |
|----------|------|------|
| sales_train_validation.csv | 30,490 | d_1〜d_1913 の販売数 (wide形式) |
| sales_train_evaluation.csv | 30,490 | d_1〜d_1941 の販売数 (評価用、+28日) |
| calendar.csv | 1,969 | 日付、曜日、イベント、SNAP情報 |
| sell_prices.csv | 6,841,121 | 店舗×商品×週の販売価格 |
| sample_submission.csv | 60,980 | validation + evaluation の提出フォーマット |

In [ ]:
# ============================================================
# SETUP CELL — 環境・認証・データ確認・読み込み
# ============================================================
import sys, os, shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (16, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

# ============================================================
# [CONFIG] 手動設定 (自動検出する場合は None のまま)
# ============================================================
USER_DATA_DIR = None   # 例: '/content/drive/MyDrive/m5'

# ============================================================
# [1] 環境検出
# ============================================================
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ
print(f"[1] Environment : {'Google Colab' if IS_COLAB else 'Local'}")

COMPETITION = 'm5-forecasting-accuracy'
DATA_FILES  = ['calendar.csv', 'sales_train_evaluation.csv',
               'sell_prices.csv', 'sample_submission.csv']

def has_all_files(d: Path) -> bool:
    return d.exists() and all((d / f).exists() for f in DATA_FILES)

# ============================================================
# [2] DATA_DIR の決定
# ============================================================
DATA_DIR = None

if USER_DATA_DIR is not None:
    DATA_DIR = Path(USER_DATA_DIR)

elif IS_COLAB:
    # 2a. /content/ をチェック (通信なし)
    for c in [Path(f'/content/{COMPETITION}'), Path('/content/data')]:
        if has_all_files(c):
            DATA_DIR = c
            break

    # 2b. Drive をマウントして確認 (通信1回、失敗時はスキップ)
    if DATA_DIR is None:
        _drive_ok = Path('/content/drive/MyDrive').exists()
        if not _drive_ok:
            try:
                print("[2] Mounting Google Drive...")
                from google.colab import drive
                drive.mount('/content/drive', force_remount=False)
                _drive_ok = True
            except Exception as _e:
                print(f"[2] Drive mount failed: {_e}")
        if _drive_ok:
            for c in [
                Path(f'/content/drive/MyDrive/kaggle/{COMPETITION}'),
                Path(f'/content/drive/MyDrive/{COMPETITION}'),
            ]:
                if has_all_files(c):
                    DATA_DIR = c
                    break

                # 2c. Drive にも見つからない → Kaggle API でダウンロード
    if DATA_DIR is None:
        DATA_DIR = Path(f'/content/{COMPETITION}')
        DATA_DIR.mkdir(parents=True, exist_ok=True)

        # access_token を Drive から取得
        _token_path = Path('/content/drive/MyDrive/.kaggle/access_token')
        if not _token_path.exists():
            raise FileNotFoundError(
                "Google Drive の マイドライブ/.kaggle/access_token が見つかりません"
            )
        _token = _token_path.read_text().strip()
        print(f"[2] access_token : {_token[:8]}... (from Drive)")

        # kaggle CLI は使わず requests で直接ダウンロード
        import requests, zipfile
        _url = f'https://www.kaggle.com/api/v1/competitions/data/download-all/{COMPETITION}'
        print(f"[3] Downloading from Kaggle API...")
        with requests.get(_url, headers={'Authorization': f'Bearer {_token}'},
                          stream=True, allow_redirects=True) as _r:
            if _r.status_code == 401:
                raise RuntimeError("401 Unauthorized — access_token が無効です")
            if _r.status_code == 403:
                raise RuntimeError("403 Forbidden — コンペのルール同意が必要です")
            _r.raise_for_status()
            _zip = DATA_DIR / f'{COMPETITION}.zip'
            _total = int(_r.headers.get('content-length', 0))
            _done = 0
            with open(_zip, 'wb') as _f:
                for _chunk in _r.iter_content(1024 * 1024):
                    _f.write(_chunk)
                    _done += len(_chunk)
                    if _total:
                        print(f"\r    {_done/1e6:.0f} / {_total/1e6:.0f} MB", end='')
        print()
        with zipfile.ZipFile(_zip) as _z:
            _z.extractall(DATA_DIR)
        _zip.unlink()
        del _token

else:
    # ローカル: 通信なし、データは手動管理
    for c in [
        Path('/mnt/c/Users/hiroyuki_nakayama/src/kaggle-with-mcp/m5-forecasting-accuracy'),
        Path('.'),
    ]:
        if has_all_files(c):
            DATA_DIR = c
            break

# ============================================================
# [3] ファイル確認
# ============================================================
assert DATA_DIR is not None, "DATA_DIR が未設定です"
assert has_all_files(DATA_DIR), \
    f"ファイルが揃っていません: {[f for f in DATA_FILES if not (DATA_DIR/f).exists()]}"

print(f"[2] DATA_DIR    : {DATA_DIR}")
for f in DATA_FILES:
    size = (DATA_DIR / f).stat().st_size / 1e6
    print(f"    {f:<40} {size:6.1f} MB")

# ============================================================
# [4] データ読み込み
# ============================================================
FIG_DIR = DATA_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)
DATA_DIR = str(DATA_DIR) + '/'
FIG_DIR  = str(FIG_DIR) + '/'

print("[3] Loading data...")
calendar = pd.read_csv(DATA_DIR + 'calendar.csv', parse_dates=['date'])
sales    = pd.read_csv(DATA_DIR + 'sales_train_evaluation.csv')
prices   = pd.read_csv(DATA_DIR + 'sell_prices.csv')

print(f"    calendar : {calendar.shape}")
print(f"    sales    : {sales.shape}")
print(f"    prices   : {prices.shape}")
print("✓ Ready")

In [ ]:
# --- 基本統計 ---
print("=== calendar ===")
print(f"期間: {calendar['date'].min()} ~ {calendar['date'].max()}")
print(f"日数: {len(calendar)}")
print(f"\nイベント数: {calendar['event_name_1'].notna().sum()} 日 (event_1), {calendar['event_name_2'].notna().sum()} 日 (event_2)")
print(f"\nイベント種類:")
print(calendar['event_type_1'].value_counts())

print("\n=== sales ===")
meta_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
d_cols = [c for c in sales.columns if c.startswith('d_')]
print(f"商品数: {len(sales)}")
print(f"日数: {len(d_cols)} (d_1 ~ d_{len(d_cols)})")
print(f"\n州別商品数:")
print(sales['state_id'].value_counts())
print(f"\nカテゴリ別商品数:")
print(sales['cat_id'].value_counts())
print(f"\n店舗別商品数:")
print(sales['store_id'].value_counts())

print("\n=== sell_prices ===")
print(f"ユニーク商品数: {prices['item_id'].nunique()}")
print(f"ユニーク店舗数: {prices['store_id'].nunique()}")
print(f"価格範囲: ${prices['sell_price'].min():.2f} ~ ${prices['sell_price'].max():.2f}")
print(f"価格中央値: ${prices['sell_price'].median():.2f}")
print(f"週数: {prices['wm_yr_wk'].nunique()}")

## Step 1: 基本統計と時系列の全体像

In [ ]:
# --- Step 1a: 全期間の合計売上 時系列プロット ---
# d_cols の合計を日別に算出
daily_total = sales[d_cols].sum(axis=0).values  # shape: (1941,)
dates = calendar['date'].values[:len(daily_total)]

fig, ax = plt.subplots(figsize=(18, 5))
ax.plot(dates, daily_total, linewidth=0.5, alpha=0.8, color='steelblue')
ax.set_title('Total Daily Sales (All Items, All Stores)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Total Units Sold')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# 年末年始を強調
for year in range(2011, 2017):
    xmas = pd.Timestamp(f'{year}-12-25')
    if xmas >= pd.Timestamp(dates[0]) and xmas <= pd.Timestamp(dates[-1]):
        ax.axvline(xmas, color='red', alpha=0.3, linestyle='--', linewidth=0.8)

ax.legend(['Daily Total', 'Christmas'], loc='upper left')
plt.tight_layout()
plt.savefig(FIG_DIR + '01_total_daily_sales.png', dpi=150)
plt.show()

print(f"売上範囲: {daily_total.min():,.0f} ~ {daily_total.max():,.0f}")
print(f"売上平均: {daily_total.mean():,.0f} ± {daily_total.std():,.0f}")

In [ ]:
# --- Step 1b: 曜日別・月別の平均売上 ---
cal_sales = pd.DataFrame({
    'date': dates,
    'total': daily_total,
})
cal_sales = cal_sales.merge(calendar[['date', 'weekday', 'wday', 'month', 'year']], on='date')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 曜日別
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_stats = cal_sales.groupby('weekday')['total'].agg(['mean', 'std']).reindex(weekday_order)
axes[0].bar(range(7), weekday_stats['mean'], yerr=weekday_stats['std'], 
            color=['#4C72B0' if d not in ['Saturday', 'Sunday'] else '#DD8452' for d in weekday_order],
            capsize=3, alpha=0.8)
axes[0].set_xticks(range(7))
axes[0].set_xticklabels([d[:3] for d in weekday_order])
axes[0].set_title('Average Daily Sales by Weekday')
axes[0].set_ylabel('Total Units Sold')

# 月別
month_stats = cal_sales.groupby('month')['total'].agg(['mean', 'std'])
axes[1].bar(range(1, 13), month_stats['mean'], yerr=month_stats['std'],
            color='steelblue', capsize=3, alpha=0.8)
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
axes[1].set_title('Average Daily Sales by Month')
axes[1].set_ylabel('Total Units Sold')

plt.tight_layout()
plt.savefig(FIG_DIR + '02_weekday_month_sales.png', dpi=150)
plt.show()

# 曜日の売上比
print("曜日別 平均売上 (週平均=1.0 基準):")
weekday_mean = cal_sales.groupby('weekday')['total'].mean().reindex(weekday_order)
for d, v in zip(weekday_order, weekday_mean / weekday_mean.mean()):
    print(f"  {d:>10}: {v:.3f}")

In [ ]:
# --- Step 1c: イベント日の売上変動 ---
cal_with_sales = calendar[['date', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']].copy()
cal_with_sales = cal_with_sales[cal_with_sales['d'].isin([f'd_{i}' for i in range(1, len(daily_total)+1)])]
cal_with_sales['total'] = daily_total

# イベント有無別
event_days = cal_with_sales[cal_with_sales['event_name_1'].notna()]
non_event_days = cal_with_sales[cal_with_sales['event_name_1'].isna()]

print(f"イベント日: {len(event_days)} 日, 平均売上: {event_days['total'].mean():,.0f}")
print(f"非イベント日: {len(non_event_days)} 日, 平均売上: {non_event_days['total'].mean():,.0f}")
print(f"差異: {(event_days['total'].mean() / non_event_days['total'].mean() - 1)*100:+.1f}%")

# イベント種別ごとの売上
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# イベントタイプ別
event_type_sales = event_days.groupby('event_type_1')['total'].agg(['mean', 'count'])
event_type_sales = event_type_sales.sort_values('mean', ascending=True)
colors = {'Cultural': '#4C72B0', 'National': '#DD8452', 'Religious': '#55A868', 'Sporting': '#C44E52'}
bar_colors = [colors.get(t, 'gray') for t in event_type_sales.index]
axes[0].barh(event_type_sales.index, event_type_sales['mean'], color=bar_colors, alpha=0.8)
axes[0].axvline(non_event_days['total'].mean(), color='black', linestyle='--', label='Non-event avg')
for i, (idx, row) in enumerate(event_type_sales.iterrows()):
    axes[0].text(row['mean'] + 500, i, f'n={int(row["count"])}', va='center', fontsize=10)
axes[0].set_title('Average Sales by Event Type')
axes[0].set_xlabel('Total Units Sold')
axes[0].legend()

# 主要イベント別 (出現2回以上)
event_name_sales = event_days.groupby('event_name_1')['total'].agg(['mean', 'count'])
event_name_sales = event_name_sales[event_name_sales['count'] >= 2].sort_values('mean', ascending=True)
axes[1].barh(range(len(event_name_sales)), event_name_sales['mean'], color='steelblue', alpha=0.8)
axes[1].set_yticks(range(len(event_name_sales)))
axes[1].set_yticklabels(event_name_sales.index, fontsize=9)
axes[1].axvline(non_event_days['total'].mean(), color='black', linestyle='--', label='Non-event avg')
axes[1].set_title('Average Sales by Event Name (n≥2)')
axes[1].set_xlabel('Total Units Sold')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR + '03_event_sales.png', dpi=150)
plt.show()

# クリスマス・サンクスギビングの売上を具体的に
for event_name in ['Christmas', 'Thanksgiving', 'SuperBowl', 'Easter']:
    ev = event_days[event_days['event_name_1'] == event_name]
    if len(ev) > 0:
        print(f"\n{event_name}: 平均売上 {ev['total'].mean():,.0f} ({(ev['total'].mean()/non_event_days['total'].mean()-1)*100:+.1f}% vs 非イベント)")
        for _, row in ev.iterrows():
            print(f"  {row['date'].strftime('%Y-%m-%d')}: {row['total']:,.0f}")

## Step 2: 階層構造（Hierarchy）の分析
州別（CA, TX, WI）、カテゴリ別（FOODS, HOBBIES, HOUSEHOLD）の売上比率と店舗ごとの特性

In [ ]:
# --- Step 2a: 州別・カテゴリ別の売上比率 ---
total_per_item = sales[d_cols].sum(axis=1)
sales['total_sales'] = total_per_item

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 州別
state_sales = sales.groupby('state_id')['total_sales'].sum()
state_colors = {'CA': '#4C72B0', 'TX': '#DD8452', 'WI': '#55A868'}
axes[0].pie(state_sales, labels=state_sales.index, autopct='%1.1f%%', 
            colors=[state_colors[s] for s in state_sales.index], startangle=90)
axes[0].set_title('Sales by State')

# カテゴリ別
cat_sales = sales.groupby('cat_id')['total_sales'].sum()
cat_colors = {'FOODS': '#C44E52', 'HOBBIES': '#8172B2', 'HOUSEHOLD': '#CCB974'}
axes[1].pie(cat_sales, labels=cat_sales.index, autopct='%1.1f%%',
            colors=[cat_colors[c] for c in cat_sales.index], startangle=90)
axes[1].set_title('Sales by Category')

# 店舗別
store_sales = sales.groupby('store_id')['total_sales'].sum().sort_values(ascending=True)
colors_store = [state_colors[s.split('_')[0]] for s in store_sales.index]
axes[2].barh(store_sales.index, store_sales.values, color=colors_store, alpha=0.8)
axes[2].set_title('Total Sales by Store')
axes[2].set_xlabel('Total Units Sold')

plt.tight_layout()
plt.savefig(FIG_DIR + '04_hierarchy_sales.png', dpi=150)
plt.show()

# 数値でも表示
print("州×カテゴリ 売上構成比:")
cross = sales.groupby(['state_id', 'cat_id'])['total_sales'].sum().unstack()
print((cross / cross.sum().sum() * 100).round(1))

In [ ]:
# --- Step 2b: 州別の日次売上推移 ---
fig, axes = plt.subplots(3, 1, figsize=(18, 10), sharex=True)

for i, state in enumerate(['CA', 'TX', 'WI']):
    state_mask = sales['state_id'] == state
    for cat in ['FOODS', 'HOBBIES', 'HOUSEHOLD']:
        cat_mask = sales['cat_id'] == cat
        daily = sales.loc[state_mask & cat_mask, d_cols].sum(axis=0).values
        # 7日移動平均でスムージング
        daily_smooth = pd.Series(daily).rolling(7, center=True).mean().values
        axes[i].plot(dates, daily_smooth, linewidth=0.8, alpha=0.8, 
                     color=cat_colors[cat], label=cat)
    axes[i].set_title(f'{state}', fontsize=13, fontweight='bold')
    axes[i].set_ylabel('Daily Sales (7d MA)')
    axes[i].legend(loc='upper left')
    axes[i].xaxis.set_major_locator(mdates.YearLocator())
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle('Daily Sales by State × Category (7-day Moving Average)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR + '05_state_category_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Step 2c: 特定1店舗の深掘り (CA_3 = 最大売上店舗) ---
# まず最大売上店舗を特定
top_store = sales.groupby('store_id')['total_sales'].sum().idxmax()
print(f"最大売上店舗: {top_store}")

store_data = sales[sales['store_id'] == top_store]

fig, axes = plt.subplots(2, 1, figsize=(18, 8), sharex=True)

# dept_id 別の推移
dept_sales = store_data.groupby('dept_id')[d_cols].sum()
dept_total = dept_sales.sum(axis=1).sort_values(ascending=False)
top_depts = dept_total.head(6).index

for dept in top_depts:
    daily = dept_sales.loc[dept].values
    daily_smooth = pd.Series(daily).rolling(28, center=True).mean().values
    axes[0].plot(dates, daily_smooth, linewidth=1.0, alpha=0.8, label=dept)

axes[0].set_title(f'{top_store}: Daily Sales by Department (28d MA)', fontsize=13)
axes[0].set_ylabel('Daily Sales')
axes[0].legend(loc='upper left', ncol=3)

# cat_id 別の積み上げ (月次集計)
monthly = store_data.copy()
# 月次に変換
cat_monthly = {}
for cat in ['FOODS', 'HOBBIES', 'HOUSEHOLD']:
    daily_vals = store_data[store_data['cat_id'] == cat][d_cols].sum(axis=0).values
    cat_monthly[cat] = daily_vals

# 28日ごとに集計して積み上げ
period = 28
n_periods = len(daily_total) // period
x_dates = dates[::period][:n_periods]
bottom = np.zeros(n_periods)

for cat, color in cat_colors.items():
    vals = np.array([cat_monthly[cat][i*period:(i+1)*period].sum() for i in range(n_periods)])
    axes[1].bar(x_dates, vals, bottom=bottom, width=20, color=color, alpha=0.8, label=cat)
    bottom += vals

axes[1].set_title(f'{top_store}: Sales by Category (4-week periods, stacked)', fontsize=13)
axes[1].set_ylabel('Total Units Sold')
axes[1].legend(loc='upper left')
axes[1].xaxis.set_major_locator(mdates.YearLocator())
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig(FIG_DIR + '06_store_deep_dive.png', dpi=150)
plt.show()

# 部門別構成比
print(f"\n{top_store} 部門別売上構成比:")
for dept, total in dept_total.items():
    print(f"  {dept:>15}: {total:>12,.0f} ({total/dept_total.sum()*100:.1f}%)")

## Step 3: 価格（Price）と在庫の動向
sell_priceの時間経過による変化、価格変更と売上数量の相関

In [ ]:
# --- Step 3a: 価格分布とカテゴリ別価格帯 ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 全体の価格分布
axes[0].hist(prices['sell_price'], bins=100, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].set_title('Sell Price Distribution (All Items)')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_xlim(0, 30)  # 大部分は$30以下

# カテゴリ別の価格分布 (item_idからcat_idを取得)
prices_with_cat = prices.merge(sales[['item_id', 'cat_id']].drop_duplicates(), on='item_id')
for cat, color in cat_colors.items():
    cat_prices = prices_with_cat[prices_with_cat['cat_id'] == cat]['sell_price']
    axes[1].hist(cat_prices, bins=80, alpha=0.5, color=color, label=f'{cat} (med=${cat_prices.median():.2f})')

axes[1].set_title('Price Distribution by Category')
axes[1].set_xlabel('Price ($)')
axes[1].set_ylabel('Frequency')
axes[1].set_xlim(0, 30)
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR + '07_price_distribution.png', dpi=150)
plt.show()

print("カテゴリ別 価格統計:")
print(prices_with_cat.groupby('cat_id')['sell_price'].describe().round(2))

In [ ]:
# --- Step 3b: サンプル商品の価格推移 ---
# 売上上位の商品から各カテゴリ2つずつサンプル
np.random.seed(42)
sample_items = []
for cat in ['FOODS', 'HOBBIES', 'HOUSEHOLD']:
    cat_items = sales[sales['cat_id'] == cat].nlargest(20, 'total_sales')['item_id'].values
    sample_items.extend(np.random.choice(cat_items, 2, replace=False))

# wm_yr_wk → 日付マッピング (週の最初の日)
wk_to_date = calendar.groupby('wm_yr_wk')['date'].first().to_dict()

fig, axes = plt.subplots(3, 2, figsize=(16, 10))
axes = axes.flatten()

for i, item in enumerate(sample_items):
    item_prices = prices[(prices['item_id'] == item) & (prices['store_id'] == top_store)]
    if len(item_prices) == 0:
        item_prices = prices[prices['item_id'] == item].groupby('wm_yr_wk')['sell_price'].mean().reset_index()
    
    item_prices = item_prices.copy()
    item_prices['date'] = item_prices['wm_yr_wk'].map(wk_to_date)
    item_prices = item_prices.dropna(subset=['date']).sort_values('date')
    
    cat = item.split('_')[0]
    axes[i].plot(item_prices['date'], item_prices['sell_price'], 
                 color=cat_colors[cat], linewidth=1.0, alpha=0.8)
    axes[i].set_title(f'{item}', fontsize=10)
    axes[i].set_ylabel('Price ($)')
    
    # 価格変更ポイントをマーク
    price_changes = item_prices['sell_price'].diff().ne(0) & item_prices['sell_price'].diff().notna()
    n_changes = price_changes.sum()
    axes[i].text(0.02, 0.95, f'{n_changes} price changes', transform=axes[i].transAxes, 
                 fontsize=9, va='top')

plt.suptitle(f'Price History of Sample Items ({top_store})', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR + '08_price_history.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Step 3c: 価格変更と売上の相関 ---
# サンプル: 売上上位100商品 × top_store で価格変更の影響を集計
top_items = sales[sales['store_id'] == top_store].nlargest(100, 'total_sales')

# 週次の売上を作る (wm_yr_wk単位)
wk_to_dcols = {}
for _, row in calendar[calendar['d'].isin(d_cols)].iterrows():
    wk = row['wm_yr_wk']
    if wk not in wk_to_dcols:
        wk_to_dcols[wk] = []
    wk_to_dcols[wk].append(row['d'])

# 各商品の価格変更前後の売上変化を集計
price_change_effects = []

for _, item_row in top_items.iterrows():
    item_id = item_row['item_id']
    item_prices = prices[(prices['item_id'] == item_id) & (prices['store_id'] == top_store)].sort_values('wm_yr_wk')
    
    if len(item_prices) < 3:
        continue
    
    item_prices = item_prices.copy()
    item_prices['price_pct_change'] = item_prices['sell_price'].pct_change()
    
    # 週別売上を計算
    weekly_sales = []
    for _, pr in item_prices.iterrows():
        wk = pr['wm_yr_wk']
        if wk in wk_to_dcols:
            cols_in_wk = [c for c in wk_to_dcols[wk] if c in d_cols]
            if cols_in_wk:
                wk_sale = item_row[cols_in_wk].sum()
                weekly_sales.append(wk_sale)
            else:
                weekly_sales.append(np.nan)
        else:
            weekly_sales.append(np.nan)
    
    item_prices['weekly_sales'] = weekly_sales
    item_prices['sales_pct_change'] = item_prices['weekly_sales'].pct_change()
    
    # 価格変更があった週のみ
    changed = item_prices[item_prices['price_pct_change'].abs() > 0.01].dropna(subset=['sales_pct_change'])
    for _, c in changed.iterrows():
        if np.isfinite(c['price_pct_change']) and np.isfinite(c['sales_pct_change']):
            price_change_effects.append({
                'price_change_pct': c['price_pct_change'] * 100,
                'sales_change_pct': c['sales_pct_change'] * 100,
                'cat': item_id.split('_')[0]
            })

effects_df = pd.DataFrame(price_change_effects)
# 外れ値除去 (±200%)
effects_df = effects_df[(effects_df['sales_change_pct'].abs() < 200) & (effects_df['price_change_pct'].abs() < 50)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 散布図
for cat, color in cat_colors.items():
    mask = effects_df['cat'] == cat
    axes[0].scatter(effects_df.loc[mask, 'price_change_pct'], 
                    effects_df.loc[mask, 'sales_change_pct'],
                    alpha=0.3, s=20, color=color, label=cat)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].axvline(0, color='black', linewidth=0.5)
axes[0].set_xlabel('Price Change (%)')
axes[0].set_ylabel('Sales Change (%)')
axes[0].set_title('Price Change vs Sales Change (Weekly)')
axes[0].legend()

# 値下げ/値上げ時の売上変化の分布
price_down = effects_df[effects_df['price_change_pct'] < -1]['sales_change_pct']
price_up = effects_df[effects_df['price_change_pct'] > 1]['sales_change_pct']

axes[1].hist(price_down, bins=50, alpha=0.6, color='green', label=f'Price Down (n={len(price_down)}, med={price_down.median():+.1f}%)')
axes[1].hist(price_up, bins=50, alpha=0.6, color='red', label=f'Price Up (n={len(price_up)}, med={price_up.median():+.1f}%)')
axes[1].axvline(0, color='black', linewidth=0.5)
axes[1].set_xlabel('Sales Change (%)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Sales Response to Price Changes')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR + '09_price_sales_correlation.png', dpi=150)
plt.show()

corr = effects_df['price_change_pct'].corr(effects_df['sales_change_pct'])
print(f"価格変更率 vs 売上変更率 相関係数: {corr:.3f}")
print(f"値下げ時の売上変化中央値: {price_down.median():+.1f}%")
print(f"値上げ時の売上変化中央値: {price_up.median():+.1f}%")

## Step 4: 間欠需要（Zero Sales）の特定
売上0の日の割合、「発売前で0」と「発売後に0」の区別、0連続日数の分布

In [ ]:
# --- Step 4a: 売上0の全体像 ---
sales_matrix = sales[d_cols].values  # shape: (30490, 1941)

total_cells = sales_matrix.size
zero_cells = (sales_matrix == 0).sum()
zero_pct = zero_cells / total_cells * 100

print(f"全セル数: {total_cells:,}")
print(f"売上0のセル数: {zero_cells:,} ({zero_pct:.1f}%)")
print(f"売上>0のセル数: {total_cells - zero_cells:,} ({100-zero_pct:.1f}%)")

# 商品別の0比率
zero_pct_per_item = (sales_matrix == 0).sum(axis=1) / sales_matrix.shape[1] * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(zero_pct_per_item, bins=100, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].set_title('Distribution of Zero-Sales Rate per Item')
axes[0].set_xlabel('% of Days with Zero Sales')
axes[0].set_ylabel('Number of Items')
axes[0].axvline(zero_pct_per_item.mean(), color='red', linestyle='--', label=f'Mean={zero_pct_per_item.mean():.1f}%')
axes[0].legend()

# カテゴリ別
for cat, color in cat_colors.items():
    mask = sales['cat_id'] == cat
    axes[1].hist(zero_pct_per_item[mask.values], bins=50, alpha=0.5, color=color, label=cat)
axes[1].set_title('Zero-Sales Rate by Category')
axes[1].set_xlabel('% of Days with Zero Sales')
axes[1].set_ylabel('Number of Items')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR + '10_zero_sales_distribution.png', dpi=150)
plt.show()

print(f"\n商品別 0比率統計:")
print(f"  平均: {zero_pct_per_item.mean():.1f}%")
print(f"  中央値: {np.median(zero_pct_per_item):.1f}%")
print(f"  0比率>90%の商品: {(zero_pct_per_item > 90).sum():,} ({(zero_pct_per_item > 90).sum()/len(zero_pct_per_item)*100:.1f}%)")
print(f"  0比率>50%の商品: {(zero_pct_per_item > 50).sum():,} ({(zero_pct_per_item > 50).sum()/len(zero_pct_per_item)*100:.1f}%)")

In [ ]:
# --- Step 4b: 「発売前で0」vs「発売後に0」の判別 ---
# 各商品の「初回販売日」= 最初に売上>0になった日のインデックス
first_sale_idx = np.argmax(sales_matrix > 0, axis=1)  # 最初の非ゼロのインデックス

# 初回販売がない(全部0)の商品
never_sold = (sales_matrix.sum(axis=1) == 0)
print(f"一度も売れていない商品: {never_sold.sum()} ({never_sold.sum()/len(never_sold)*100:.2f}%)")

# 発売前の0と発売後の0を分離
pre_launch_zeros = 0
post_launch_zeros = 0
post_launch_total_days = 0

for i in range(len(sales_matrix)):
    if never_sold[i]:
        pre_launch_zeros += sales_matrix.shape[1]
        continue
    
    first = first_sale_idx[i]
    pre_launch_zeros += first  # 初回販売前の0
    post_launch = sales_matrix[i, first:]
    post_launch_zeros += (post_launch == 0).sum()
    post_launch_total_days += len(post_launch)

print(f"\n発売前の0: {pre_launch_zeros:,} ({pre_launch_zeros/zero_cells*100:.1f}% of all zeros)")
print(f"発売後の0: {post_launch_zeros:,} ({post_launch_zeros/zero_cells*100:.1f}% of all zeros)")
print(f"発売後の0比率: {post_launch_zeros/post_launch_total_days*100:.1f}% (発売後の全日数に対して)")

# 初回販売日の分布
fig, ax = plt.subplots(figsize=(16, 5))
first_sale_dates = [dates[idx] for idx, ns in zip(first_sale_idx, never_sold) if not ns]
ax.hist(first_sale_dates, bins=100, color='steelblue', alpha=0.7, edgecolor='white')
ax.set_title('Distribution of First Sale Date')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Items')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig(FIG_DIR + '11_first_sale_date.png', dpi=150)
plt.show()

# 初回販売日がd_1の商品 vs 途中参入
day1_items = (first_sale_idx == 0) & ~never_sold
print(f"\nd_1から存在する商品: {day1_items.sum()} ({day1_items.sum()/len(day1_items)*100:.1f}%)")
print(f"途中参入の商品: {(~day1_items & ~never_sold).sum()} ({(~day1_items & ~never_sold).sum()/len(day1_items)*100:.1f}%)")

In [ ]:
# --- Step 4c: 発売後の0連続日数の分布 ---
from itertools import groupby

# サンプリング: ランダム3000商品 (全商品は重いので)
np.random.seed(42)
sample_idx = np.random.choice(np.where(~never_sold)[0], size=min(3000, (~never_sold).sum()), replace=False)

zero_streak_lengths = []

for i in sample_idx:
    first = first_sale_idx[i]
    post_launch = sales_matrix[i, first:]
    
    # 連続0のストリーク長を計算
    for is_zero, group in groupby(post_launch == 0):
        if is_zero:
            streak_len = sum(1 for _ in group)
            zero_streak_lengths.append(streak_len)

zero_streaks = np.array(zero_streak_lengths)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 連続0日数の分布 (ヒストグラム)
axes[0].hist(zero_streaks[zero_streaks <= 60], bins=60, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].set_title('Distribution of Consecutive Zero-Sales Streaks (≤60 days)')
axes[0].set_xlabel('Consecutive Zero Days')
axes[0].set_ylabel('Frequency')
axes[0].axvline(7, color='red', linestyle='--', alpha=0.5, label='1 week')
axes[0].axvline(28, color='orange', linestyle='--', alpha=0.5, label='4 weeks')
axes[0].legend()

# 累積分布
sorted_streaks = np.sort(zero_streaks)
axes[1].plot(sorted_streaks, np.arange(1, len(sorted_streaks)+1)/len(sorted_streaks), 
             color='steelblue', linewidth=1.5)
axes[1].set_title('CDF of Consecutive Zero-Sales Streaks')
axes[1].set_xlabel('Consecutive Zero Days')
axes[1].set_ylabel('Cumulative Proportion')
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].axhline(0.9, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlim(0, 200)

# パーセンタイルをアノテーション
for p, label in [(50, '50th'), (90, '90th'), (99, '99th')]:
    val = np.percentile(zero_streaks, p)
    axes[1].annotate(f'{label}: {val:.0f}d', xy=(val, p/100), fontsize=10,
                     xytext=(val+10, p/100-0.05),
                     arrowprops=dict(arrowstyle='->', color='gray'))

plt.tight_layout()
plt.savefig(FIG_DIR + '12_zero_streak_distribution.png', dpi=150)
plt.show()

print(f"0連続日数の統計 (発売後, n={len(zero_streaks):,} streaks):")
print(f"  平均: {zero_streaks.mean():.1f} 日")
print(f"  中央値: {np.median(zero_streaks):.1f} 日")
print(f"  90th percentile: {np.percentile(zero_streaks, 90):.0f} 日")
print(f"  99th percentile: {np.percentile(zero_streaks, 99):.0f} 日")
print(f"  最大: {zero_streaks.max()} 日")

# 長期ゼロ (28日以上) の割合
long_zero = (zero_streaks >= 28).sum()
print(f"\n28日以上連続0: {long_zero:,} streaks ({long_zero/len(zero_streaks)*100:.1f}%)")
print(f"  → 在庫切れ or 廃番の可能性が高い")

In [ ]:
# --- Step 4d: 日別の0比率推移 (発売後のみ) ---
# 各日の「発売済み商品数」と「そのうち売上0の商品数」
active_items_per_day = np.zeros(len(d_cols))
zero_active_per_day = np.zeros(len(d_cols))

for i in range(len(sales_matrix)):
    if never_sold[i]:
        continue
    first = first_sale_idx[i]
    for j in range(first, len(d_cols)):
        active_items_per_day[j] += 1
        if sales_matrix[i, j] == 0:
            zero_active_per_day[j] += 1

zero_rate_per_day = zero_active_per_day / np.maximum(active_items_per_day, 1) * 100

fig, axes = plt.subplots(2, 1, figsize=(18, 8), sharex=True)

# 発売済み商品数の推移
axes[0].plot(dates, active_items_per_day, linewidth=0.8, color='steelblue')
axes[0].set_title('Number of Active Items (Post-Launch) Over Time')
axes[0].set_ylabel('Active Items')

# 発売後0比率の推移
axes[1].plot(dates, zero_rate_per_day, linewidth=0.5, color='tomato', alpha=0.5)
# 28日移動平均
zero_rate_smooth = pd.Series(zero_rate_per_day).rolling(28, center=True).mean().values
axes[1].plot(dates, zero_rate_smooth, linewidth=1.5, color='red', label='28d MA')
axes[1].set_title('Zero-Sales Rate Among Active Items')
axes[1].set_ylabel('Zero-Sales Rate (%)')
axes[1].set_xlabel('Date')
axes[1].legend()
axes[1].xaxis.set_major_locator(mdates.YearLocator())
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig(FIG_DIR + '13_zero_rate_timeseries.png', dpi=150)
plt.show()

print(f"発売後0比率: 平均 {zero_rate_per_day.mean():.1f}%, 最小 {zero_rate_per_day.min():.1f}%, 最大 {zero_rate_per_day.max():.1f}%")

## Step 5: 店舗プロファイリング — 商圏人口スコア & 贅沢品指数
必需品（FOODS_3 の低CV×高販売数アイテム）をアンカーに商圏規模を推定し、高価格帯商品の購買密度から店舗の所得特性を明らかにする

In [ ]:
# --- Step 5a: アンカーアイテムの特定 (FOODS_3 内 低CV×高販売数 Top5) ---
foods3 = sales[sales['dept_id'] == 'FOODS_3']
foods3_daily = foods3.groupby('item_id')[d_cols].sum()  # item_id × days

foods3_stats = pd.DataFrame({
    'daily_mean': foods3_daily.mean(axis=1),
    'cv': foods3_daily.std(axis=1) / foods3_daily.mean(axis=1),
})

# 平均販売数が中央値以上 かつ CV が小さい順
median_sales = foods3_stats['daily_mean'].median()
candidates = foods3_stats[foods3_stats['daily_mean'] >= median_sales].sort_values('cv')
anchor_items = candidates.head(5).index.tolist()

print("=== アンカーアイテム (FOODS_3: 低CV × 高販売数 Top5) ===")
print(candidates.head(5).to_string())
print(f"\nAnchor items: {anchor_items}")

In [ ]:
# --- Step 5b: 商圏人口スコア & 贅沢品指数の算出 ---

# 1) 店舗別アンカー商品の1日平均販売数 → 商圏人口スコア
anchor_sales = sales[sales['item_id'].isin(anchor_items)]
store_anchor_daily = anchor_sales.groupby('store_id')[d_cols].sum().mean(axis=1)
overall_mean = store_anchor_daily.mean()
market_score = (store_anchor_daily / overall_mean).sort_values(ascending=False)

# 2) 高価格帯商品の特定 (各カテゴリ上位25%)
item_avg_price = prices.groupby('item_id')['sell_price'].mean()
item_cat = sales[['item_id', 'cat_id']].drop_duplicates().set_index('item_id')['cat_id']
price_df = pd.DataFrame({'avg_price': item_avg_price, 'cat_id': item_cat}).dropna()

luxury_items = set()
print("=== 高価格帯商品 (各カテゴリ 75%ile以上) ===")
for cat, grp in price_df.groupby('cat_id'):
    threshold = grp['avg_price'].quantile(0.75)
    top = grp[grp['avg_price'] >= threshold].index.tolist()
    luxury_items.update(top)
    print(f"  {cat}: 閾値=${threshold:.2f}, {len(top)} items")

# 3) 店舗別贅沢品の1日平均販売数 / 商圏人口スコア
lux_sales = sales[sales['item_id'].isin(luxury_items)]
store_lux_daily = lux_sales.groupby('store_id')[d_cols].sum().mean(axis=1)
luxury_index = store_lux_daily / market_score

# まとめ DataFrame
profile = pd.DataFrame({
    'market_score': market_score,
    'anchor_daily_avg': store_anchor_daily,
    'luxury_daily_avg': store_lux_daily,
    'luxury_index': luxury_index,
}).sort_values('luxury_index', ascending=False)
profile['state'] = profile.index.map(lambda x: x.split('_')[0])

print("\n=== 店舗プロファイル ===")
print(profile.to_string())

In [ ]:
# --- Step 5c: 店舗プロファイル 可視化 ---
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

state_colors = {'CA': '#4C72B0', 'TX': '#DD8452', 'WI': '#55A868'}
bar_colors = [state_colors[s] for s in profile['state']]

# (1) 商圏人口スコア
ax = axes[0, 0]
ax.barh(profile.index, profile['market_score'], color=bar_colors, alpha=0.8)
ax.axvline(1.0, color='black', linestyle='--', linewidth=1, label='Average (1.0)')
ax.set_xlabel('Market Size Score (avg=1.0)')
ax.set_title('Market Size Score (Anchor Item Sales)')
ax.legend()
for i, (store, row) in enumerate(profile.iterrows()):
    ax.text(row['market_score'] + 0.02, i, f"{row['market_score']:.2f}", va='center', fontsize=10)

# (2) 贅沢品指数
ax = axes[0, 1]
lux_sorted = profile.sort_values('luxury_index', ascending=True)
lux_colors = [state_colors[s] for s in lux_sorted['state']]
ax.barh(lux_sorted.index, lux_sorted['luxury_index'], color=lux_colors, alpha=0.8)
ax.set_xlabel('Luxury Index (luxury sales / market score)')
ax.set_title('Luxury Index (Income Proxy)')
for i, (store, row) in enumerate(lux_sorted.iterrows()):
    ax.text(row['luxury_index'] + 5, i, f"{row['luxury_index']:.0f}", va='center', fontsize=10)

# (3) 散布図: 商圏スコア vs 贅沢品指数
ax = axes[1, 0]
for state, color in state_colors.items():
    mask = profile['state'] == state
    ax.scatter(profile.loc[mask, 'market_score'], profile.loc[mask, 'luxury_index'],
               color=color, s=150, alpha=0.8, label=state, edgecolors='black', linewidth=0.5)
    for store in profile.loc[mask].index:
        ax.annotate(store, (profile.loc[store, 'market_score'], profile.loc[store, 'luxury_index']),
                    textcoords='offset points', xytext=(8, 4), fontsize=9)
ax.set_xlabel('Market Size Score')
ax.set_ylabel('Luxury Index')
ax.set_title('Market Size vs Luxury Index')
ax.legend(title='State')
ax.axvline(1.0, color='gray', linestyle='--', alpha=0.3)
ax.axhline(profile['luxury_index'].median(), color='gray', linestyle='--', alpha=0.3)

# (4) カテゴリ別売上構成比 (店舗ごと、贅沢品指数順)
ax = axes[1, 1]
cat_colors_ordered = {'FOODS': '#C44E52', 'HOUSEHOLD': '#CCB974', 'HOBBIES': '#8172B2'}
store_order = profile.sort_values('luxury_index', ascending=False).index

cat_pcts = {}
for cat in ['FOODS', 'HOUSEHOLD', 'HOBBIES']:
    cat_store_sales = sales[sales['cat_id'] == cat].groupby('store_id')[d_cols].sum().sum(axis=1)
    cat_pcts[cat] = cat_store_sales

cat_pct_df = pd.DataFrame(cat_pcts).loc[store_order]
cat_pct_df = cat_pct_df.div(cat_pct_df.sum(axis=1), axis=0) * 100

bottom = np.zeros(len(store_order))
for cat in ['FOODS', 'HOUSEHOLD', 'HOBBIES']:
    vals = cat_pct_df[cat].values
    ax.barh(store_order, vals, left=bottom, color=cat_colors_ordered[cat], alpha=0.8, label=cat)
    bottom += vals
ax.set_xlabel('Sales Share (%)')
ax.set_title('Category Mix by Store (sorted by Luxury Index)')
ax.legend(loc='lower right')

plt.suptitle('Step 5: Store Profiling — Market Size & Luxury Index', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR + '14_store_profiling.png', dpi=150, bbox_inches='tight')
plt.show()

# 州別サマリ
print("\n=== 州別サマリ ===")
for state in ['CA', 'TX', 'WI']:
    s = profile[profile['state'] == state]
    print(f"  {state}: 商圏スコア平均={s['market_score'].mean():.2f}, 贅沢品指数平均={s['luxury_index'].mean():.0f}")

## Step 6: 来訪者・非日常消費の徹底解剖
品揃えの物理制約、平日密度による外部流入検知、イベント属性の二分化、曜日vsイベントの分散分解、「せっかく買い」指数

In [ ]:
# --- Step 6a: 品揃え(Assortment)の物理的検証 ---
# 各店舗で1日でも販売実績がある item_id を dept_id ごとに集計
sales['ever_sold'] = (sales[d_cols].values > 0).any(axis=1)
assortment = sales[sales['ever_sold']].groupby(['store_id', 'dept_id'])['item_id'].nunique().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(assortment, annot=True, fmt='d', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Active Item Count by Store x Department', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR + '21_assortment.png', dpi=150)
plt.show()

# 全店舗で完全に同一の品揃え
print(f"全店舗の品揃え差: {(assortment.max() - assortment.min()).sum()} (0 = 完全同一)")
print(f"結論: 全10店舗で品揃え完全同一 (各{assortment.sum(axis=1).iloc[0]}品)")
print("→ Luxury Index の差は品揃えではなく、純粋な購買行動の差")

In [ ]:
# --- Step 6b: 平日密度 (Mon-Thu / Fri-Sun) — 外部流入シグナル ---
cal_wd = calendar.set_index('d')['weekday'].to_dict()
d_weekdays = [cal_wd.get(d, '') for d in d_cols]
mon_thu = [i for i, w in enumerate(d_weekdays) if w in ['Monday','Tuesday','Wednesday','Thursday']]
fri_sun = [i for i, w in enumerate(d_weekdays) if w in ['Friday','Saturday','Sunday']]

depts = sorted(sales['dept_id'].unique())
stores_sorted = sorted(sales['store_id'].unique())
ratios = {}
for store in stores_sorted:
    ratios[store] = {}
    for dept in depts:
        daily = sales.loc[(sales['store_id']==store) & (sales['dept_id']==dept), d_cols].sum(axis=0).values
        ratios[store][dept] = daily[mon_thu].mean() / daily[fri_sun].mean()

ratio_df = pd.DataFrame(ratios).T
ratio_mean = ratio_df.mean()
ratio_diff = ratio_df.sub(ratio_mean, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
sns.heatmap(ratio_df, annot=True, fmt='.3f', cmap='RdYlBu_r', center=ratio_df.values.mean(),
            ax=axes[0], linewidths=0.5, vmin=0.6, vmax=0.9)
axes[0].set_title('Weekday/Weekend Ratio (Higher = stronger weekday)', fontsize=12)
sns.heatmap(ratio_diff, annot=True, fmt='.3f', cmap='RdBu_r', center=0, ax=axes[1], linewidths=0.5)
axes[1].set_title('Diff from Average (Red = anomalous weekday density)', fontsize=12)
plt.suptitle('Step 6b: Weekday Density — External Visitor Signal', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR + '22_weekday_density.png', dpi=150, bbox_inches='tight')
plt.show()

# 上位の異常値
print("=== 平日密度が特に高い (上位5) ===")
flat = ratio_diff.stack().sort_values(ascending=False).head(5)
for (store, dept), v in flat.items():
    print(f"  {store} x {dept}: ratio={ratios[store][dept]:.3f}, diff={v:+.3f}")

In [ ]:
# --- Step 6c: イベント来訪者属性 (Tourism vs Family/Local) ---
high_stores = ['CA_4', 'CA_2', 'CA_3', 'CA_1']
low_stores = ['TX_2', 'TX_3', 'WI_3', 'WI_1']

cal_ev = calendar[['d','event_name_1','event_type_1']].copy()
d_to_idx = {d: i for i, d in enumerate(d_cols)}
non_event_idx = [d_to_idx[d] for d in d_cols if d not in set(cal_ev[cal_ev['event_name_1'].notna()]['d'])]

# イベントを Tourism型 / Family型 に分類
tourism_events = ['SuperBowl', 'ValentinesDay', 'PresidentsDay',
                  'MartinLutherKingDay', 'StPatricksDay', 'CincoDeMAyo']
family_events = ['Christmas', 'Thanksgiving', 'Easter', 'Halloween',
                 'IndependenceDay', 'MemorialDay', 'LaborDay',
                 'NewYear', 'ColumbusDay', 'VeteransDay']

# 代表 dept (各カテゴリ最大売上 dept)
rep_depts = ['FOODS_2', 'HOBBIES_1', 'HOUSEHOLD_2']

def calc_event_lift(store_list, event_list, depts):
    """イベント別×dept別のリフト率を計算"""
    results = []
    for ev in event_list:
        ev_days = cal_ev[cal_ev['event_name_1'] == ev]['d'].tolist()
        ev_idx = [d_to_idx[d] for d in ev_days if d in d_to_idx]
        if not ev_idx:
            continue
        row = {'event': ev}
        for dept in depts:
            daily = sales.loc[(sales['store_id'].isin(store_list)) &
                              (sales['dept_id'] == dept), d_cols].sum(axis=0).values
            base = daily[non_event_idx].mean()
            ev_mean = daily[ev_idx].mean()
            row[dept] = (ev_mean / base - 1) * 100 if base > 0 else 0
        results.append(row)
    return pd.DataFrame(results).set_index('event')

hi_tour = calc_event_lift(high_stores, tourism_events, rep_depts)
hi_fam  = calc_event_lift(high_stores, family_events, rep_depts)
lo_tour = calc_event_lift(low_stores, tourism_events, rep_depts)
lo_fam  = calc_event_lift(low_stores, family_events, rep_depts)

fig, axes = plt.subplots(2, 2, figsize=(20, 14))
panels = [
    (axes[0,0], hi_tour, 'High Income — Tourism Events (Total Lift %)'),
    (axes[0,1], hi_fam,  'High Income — Family Events (Total Lift %)'),
    (axes[1,0], lo_tour, 'Low Income — Tourism Events (Total Lift %)'),
    (axes[1,1], lo_fam,  'Low Income — Family Events (Total Lift %)'),
]
for ax, df, title in panels:
    sns.heatmap(df, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
                ax=ax, linewidths=0.5, cbar_kws={'shrink': 0.8})
    ax.set_title(title, fontsize=12)
    ax.set_ylabel('event')
    ax.set_xlabel('dept')

plt.suptitle('Analysis 3: Event Lift by Visitor Type (Tourism vs Local)', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR + '23_visitor_attribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("=== Tourism型: SuperBowl が FOODS で最大リフト (高所得+19.9%, 低所得+29.3%) ===")
print("=== Family型: Christmas/Thanksgiving は全面的に大幅減 (閉店/買い控え) ===")
print("=== ValentinesDay: 高所得FOODS -19% vs 低所得FOODS +11% → 所得層で反応が逆転 ===")

In [ ]:
# --- Step 6c: イベント来訪者属性 + 曜日vsイベント分散分解 ---
high_stores = ['CA_4', 'CA_2', 'CA_3', 'CA_1']
low_stores = ['TX_2', 'TX_3', 'WI_3', 'WI_1']
cal_ev = calendar[['d','date','weekday','wday','event_name_1','event_type_1']].copy()
cal_ev['is_weekend'] = cal_ev['weekday'].isin(['Friday','Saturday','Sunday']).astype(int)
cal_ev['has_event'] = cal_ev['event_name_1'].notna().astype(int)
d_to_idx = {d: i for i, d in enumerate(d_cols)}

# 分散分解: wday R² vs event marginal R²
var_results = []
for store in stores_sorted:
    for cat in ['FOODS', 'HOBBIES', 'HOUSEHOLD']:
        daily = sales.loc[(sales['store_id']==store) & (sales['cat_id']==cat), d_cols].sum(axis=0).values
        wday_vals = cal_ev['wday'].values[:len(d_cols)]
        event_vals = cal_ev['has_event'].values[:len(d_cols)]
        total_var = np.var(daily)
        # wday R²
        wday_means = np.array([daily[wday_vals==w].mean() for w in range(1,8)])
        wday_pred = np.array([wday_means[w-1] for w in wday_vals])
        wday_r2 = 1 - np.var(daily - wday_pred) / total_var
        # event R²
        ev_means = np.array([daily[event_vals==e].mean() for e in [0,1]])
        ev_pred = np.array([ev_means[e] for e in event_vals])
        event_r2 = 1 - np.var(daily - ev_pred) / total_var
        # combined
        from itertools import product
        combos = {}
        for w, e in product(range(1,8), [0,1]):
            mask = (wday_vals==w) & (event_vals==e)
            if mask.sum() > 0: combos[(w,e)] = daily[mask].mean()
        combo_pred = np.array([combos.get((w,e), daily.mean()) for w,e in zip(wday_vals, event_vals)])
        combo_r2 = 1 - np.var(daily - combo_pred) / total_var
        var_results.append({
            'store': store, 'cat': cat, 'state': store.split('_')[0],
            'wday_r2': wday_r2, 'event_marginal': combo_r2 - wday_r2,
        })

vr = pd.DataFrame(var_results)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
# Wday R²
pivot_w = vr.pivot_table(index='store', columns='cat', values='wday_r2')
sns.heatmap(pivot_w, annot=True, fmt='.4f', cmap='Blues', ax=axes[0], linewidths=0.5)
axes[0].set_title('Weekday R²')
# Event marginal R²
pivot_e = vr.pivot_table(index='store', columns='cat', values='event_marginal')
sns.heatmap(pivot_e, annot=True, fmt='.4f', cmap='Oranges', ax=axes[1], linewidths=0.5)
axes[1].set_title('Event Marginal R² (additional over weekday)')
# Scatter
ax = axes[2]
from matplotlib.lines import Line2D
for cat, marker in [('FOODS','o'),('HOBBIES','s'),('HOUSEHOLD','^')]:
    sub = vr[vr['cat']==cat]
    for _, r in sub.iterrows():
        ax.scatter(r['wday_r2'], r['event_marginal'], color=state_colors[r['state']], marker=marker, s=100, alpha=0.7)
legend_el = [Line2D([0],[0],marker='o',color='w',markerfacecolor='gray',label='FOODS',markersize=8),
             Line2D([0],[0],marker='s',color='w',markerfacecolor='gray',label='HOBBIES',markersize=8),
             Line2D([0],[0],marker='^',color='w',markerfacecolor='gray',label='HOUSEHOLD',markersize=8)]
for s, c in state_colors.items():
    legend_el.append(Line2D([0],[0],marker='o',color='w',markerfacecolor=c,label=s,markersize=8))
ax.legend(handles=legend_el, fontsize=8)
ax.set_xlabel('Weekday R²'); ax.set_ylabel('Event Marginal R²')
ax.set_title('Weekday vs Event Importance')
plt.suptitle('Step 6c: Variance Decomposition — Weekday vs Event', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR + '24_variance_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

# CA_4 のイベント重要度が突出
print("=== CA_4: 曜日の説明力が最弱、イベントの追加説明力が最強 ===")
ca4 = vr[vr['store']=='CA_4']
for _, r in ca4.iterrows():
    print(f"  {r['cat']}: wday_r2={r['wday_r2']:.4f}, event_marginal={r['event_marginal']:.4f}, ratio={r['event_marginal']/r['wday_r2']:.2f}")

In [ ]:
# --- Step 6d: 「せっかく買い」指数 — イベント時のPremium品購買 ---
# Premium品 = 各カテゴリ上位10%価格
item_avg_price = prices.groupby('item_id')['sell_price'].mean()
item_cat_map = sales[['item_id','cat_id']].drop_duplicates().set_index('item_id')['cat_id']
price_info = pd.DataFrame({'avg_price': item_avg_price, 'cat_id': item_cat_map}).dropna()

premium_items = set()
for cat, grp in price_info.groupby('cat_id'):
    threshold = grp['avg_price'].quantile(0.90)
    premium_items.update(grp[grp['avg_price'] >= threshold].index)

event_d = set(cal_ev[cal_ev['event_name_1'].notna()]['d']) & set(d_cols)
non_event_d = set(d_cols) - event_d
event_idx = [d_cols.index(d) for d in event_d]
non_event_idx = [d_cols.index(d) for d in non_event_d]

# クラスタ × カテゴリ別
impulse_results = []
for cn, cs in [('High Income', high_stores), ('Low Income', low_stores)]:
    for cat in ['FOODS', 'HOBBIES', 'HOUSEHOLD']:
        prem = sales[(sales['store_id'].isin(cs)) & (sales['cat_id']==cat) & (sales['item_id'].isin(premium_items))]
        non = sales[(sales['store_id'].isin(cs)) & (sales['cat_id']==cat) & (~sales['item_id'].isin(premium_items))]
        dp = prem[d_cols].sum(axis=0).values
        dn = non[d_cols].sum(axis=0).values
        pl = (dp[event_idx].mean() / dp[non_event_idx].mean() - 1) * 100
        nl = (dn[event_idx].mean() / dn[non_event_idx].mean() - 1) * 100
        impulse_results.append({'cluster': cn, 'cat': cat, 'prem_lift': pl, 'non_prem_lift': nl, 'impulse': pl - nl})

imp_df = pd.DataFrame(impulse_results)

# イベント別 (HOBBIES × High Income)
major_events = ['SuperBowl','Easter','MemorialDay','IndependenceDay','LaborDay','Halloween','Thanksgiving']
hp = sales[(sales['store_id'].isin(high_stores)) & (sales['cat_id']=='HOBBIES') & (sales['item_id'].isin(premium_items))]
hn = sales[(sales['store_id'].isin(high_stores)) & (sales['cat_id']=='HOBBIES') & (~sales['item_id'].isin(premium_items))]
hp_d = hp[d_cols].sum(axis=0).values; hn_d = hn[d_cols].sum(axis=0).values
bp = hp_d[non_event_idx].mean(); bn = hn_d[non_event_idx].mean()

ev_imp = []
for ev in major_events:
    ev_i = [d_cols.index(d) for d in cal_ev[cal_ev['event_name_1']==ev]['d'] if d in d_to_idx]
    if ev_i:
        pl = (hp_d[ev_i].mean()/bp-1)*100; nl = (hn_d[ev_i].mean()/bn-1)*100
        ev_imp.append({'event': ev, 'prem_lift': pl, 'non_prem_lift': nl, 'impulse': pl-nl})
ev_imp_df = pd.DataFrame(ev_imp)

fig, axes = plt.subplots(2, 2, figsize=(20, 12))

# (a) クラスタ×カテゴリ
ax = axes[0, 0]
x = np.arange(3); w = 0.35
hi = imp_df[imp_df['cluster']=='High Income']['impulse'].values
lo = imp_df[imp_df['cluster']=='Low Income']['impulse'].values
ax.bar(x-w/2, hi, w, label='High Income (CA)', color='steelblue', alpha=0.8)
ax.bar(x+w/2, lo, w, label='Low Income (TX/WI)', color='coral', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(['FOODS','HOBBIES','HOUSEHOLD'])
ax.set_ylabel('Impulse Index (%)'); ax.set_title('Impulse Buy Index by Cluster x Category')
ax.legend(); ax.axhline(0, color='gray', linestyle='--', alpha=0.3)

# (b) Premium vs Non-Premium
ax = axes[0, 1]
for i, cl in enumerate(['High Income','Low Income']):
    sub = imp_df[imp_df['cluster']==cl]
    xp = np.arange(3) + i*0.4 - 0.2
    co = 'steelblue' if i==0 else 'coral'
    ax.scatter(xp, sub['prem_lift'], marker='D', s=100, color=co, label=f'{cl} Premium')
    ax.scatter(xp, sub['non_prem_lift'], marker='o', s=100, alpha=0.5, color=co, label=f'{cl} Non-Premium')
    for j, (_, r) in enumerate(sub.iterrows()):
        ax.plot([xp[j],xp[j]], [r['non_prem_lift'],r['prem_lift']], 'k-', alpha=0.3)
ax.set_xticks(range(3)); ax.set_xticklabels(['FOODS','HOBBIES','HOUSEHOLD'])
ax.set_ylabel('Event Day Lift (%)'); ax.set_title('Premium vs Non-Premium Event Lift')
ax.legend(fontsize=8)

# (c) イベント別せっかく買い
ax = axes[1, 0]
es = ev_imp_df.sort_values('impulse', ascending=True)
colors_imp = ['#e74c3c' if v>0 else '#3498db' for v in es['impulse']]
ax.barh(es['event'], es['impulse'], color=colors_imp, alpha=0.8)
ax.set_xlabel('Impulse Index (%)'); ax.set_title('HOBBIES x High Income: Impulse by Event')
ax.axvline(0, color='black', linewidth=0.5)

# (d) Prem/NonPrem比較
ax = axes[1, 1]
es2 = ev_imp_df.sort_values('prem_lift', ascending=True)
y = np.arange(len(es2))
ax.barh(y-0.2, es2['prem_lift'], 0.35, label='Premium', color='gold', alpha=0.8)
ax.barh(y+0.2, es2['non_prem_lift'], 0.35, label='Non-Premium', color='silver', alpha=0.8)
ax.set_yticks(y); ax.set_yticklabels(es2['event'])
ax.set_xlabel('Event Day Lift (%)'); ax.set_title('HOBBIES x High Income: Premium vs Non-Premium')
ax.legend(); ax.axvline(0, color='black', linewidth=0.5)

plt.suptitle('Step 6d: "Impulse Buy" Index — Event-Driven Premium Purchases', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR + '25_impulse_buy.png', dpi=150, bbox_inches='tight')
plt.show()

print("=== Easter が最大のせっかく買いイベント (impulse=+36.7%) ===")
print("Premium品が+30%リフト、Non-Premium品は-7%減少 → 非日常消費が集中")
print("\n=== Independence Day/Halloween は逆にPremium品が大幅減 ===")
print("→ これらのイベントは「自宅パーティー型」で高額趣味品は不要")

## EDA サマリ

### Step 1: 時系列の全体像
- 全体の売上トレンド、曜日効果、月次季節性
- イベント日 (Christmas, Thanksgiving等) の売上変動パターン

### Step 2: 階層構造
- 州別 (CA/TX/WI)、カテゴリ別 (FOODS/HOBBIES/HOUSEHOLD) の構成比
- 店舗ごとの部門別売上推移

### Step 3: 価格と売上の関係
- カテゴリ別の価格帯
- 価格変更時の売上への影響 (価格弾力性)

### Step 4: 間欠需要
- 売上0が全データの大半を占める
- 「発売前の0」と「発売後の0」の判別
- 0連続日数の分布と長期ゼロの特定

### Step 5: 店舗プロファイリング
- **商圏人口スコア**: 必需品(FOODS_3 低CV Top5)の販売数で商圏規模を推定
  - TX_2, WI_3 が大商圏 / CA_4, CA_2 が小商圏
- **贅沢品指数**: 高価格帯商品(各カテゴリ上位25%)の販売数 / 商圏スコア
  - CA全店が Top4独占 → 高所得エリア
  - TX, WI は大商圏だが庶民的
- 散布図で商圏規模と所得特性の関係を可視化
- カテゴリ構成比と所得特性の対応を確認

### Step 6: 来訪者・非日常消費の徹底解剖
- **品揃え検証**: 全10店舗で完全同一(3,049品) → 売上差は純粋な購買行動の差
- **平日密度**: Mon-Thu/Fri-Sun比率で外部流入を検知。CA_1のHOBBIES_1が最も平日寄り(+0.029)
- **分散分解**: 曜日R²はFOODS最大(0.08-0.25)、CA_4は曜日R²最弱(0.034-0.253)だがイベント追加R²最強
  - CA_4×HOBBIES: event_marginal/wday_r2 = 0.33 (他店は0.01-0.04) → 極端なイベント駆動型
- **せっかく買い指数**: Easter が最大(+36.7%)、Premium品+30%リフト vs Non-Premium -7%
  - Independence Day/Halloween はPremium品が大幅減 → 自宅パーティー型イベント
  - 高所得クラスタ(CA)がHOBBIESで顕著にせっかく買い傾向

## Step 7: イベント普遍性 & ラマダン深層分析

### 7a: 全米共通イベントの「ハレの日」バイアス — 高単価シフト（P75+）の定量化
### 7b: ラマダン期間の価格帯別リフト — ramadan_sensitive スコア

In [ ]:
# --- Step 7a: イベント普遍性 vs 地域特異性 ---
import warnings; warnings.filterwarnings('ignore')
from collections import defaultdict

# === 価格データ準備 (sell_prices チャンク読み) ===
print("=== 価格四分位の算出 ===")
item_price_acc = {}
for chunk in pd.read_csv(DATA_DIR + 'sell_prices.csv', chunksize=500_000):
    for item, price in zip(chunk['item_id'], chunk['sell_price']):
        if item not in item_price_acc:
            item_price_acc[item] = [0.0, 0]
        item_price_acc[item][0] += price
        item_price_acc[item][1] += 1
item_avg = {k: v[0]/v[1] for k, v in item_price_acc.items() if v[1] > 0}
del item_price_acc

# dept別四分位閾値
dept_prices_map = defaultdict(list)
for item, price in item_avg.items():
    dept_prices_map['_'.join(item.split('_')[:2])].append(price)
dept_q = {d: np.percentile(p, [25, 50, 75]) for d, p in dept_prices_map.items()}
del dept_prices_map
print(f"  Dept P75: { {k: f'${v[2]:.2f}' for k,v in sorted(dept_q.items())} }")

# item → tier (0: <P25, 1: P25-50, 2: P50-75, 3: P75+)
def _get_tier(item):
    dept = '_'.join(item.split('_')[:2])
    q = dept_q.get(dept, [0, 0, 999])
    p = item_avg.get(item, 0)
    if p >= q[2]: return 3
    if p >= q[1]: return 2
    if p >= q[0]: return 1
    return 0

sales['_tier'] = sales['item_id'].map(_get_tier).astype('int8')

# === 主要イベント × 店舗のリフト計算 ===
target_events = [
    'Christmas', 'Easter', 'Thanksgiving', 'SuperBowl', 'IndependenceDay',
    'Halloween', 'LaborDay', 'MemorialDay',
    'Ramadan starts', 'Eid al-Fitr', 'EidAlAdha',
    'OrthodoxEaster', 'OrthodoxChristmas', 'Pesach End', 'Chanukah End', 'LentStart',
]
stores = sorted(sales['store_id'].unique())
d_to_idx = {d: i for i, d in enumerate(d_cols)}
ev_d_set = set(calendar[calendar['event_name_1'].notna()]['d'])
non_ev_idx = np.array([d_to_idx[d] for d in d_cols if d not in ev_d_set])

# Precompute per-store FOODS daily totals (全体 & P75+)
foods = sales['cat_id'] == 'FOODS'
store_all, store_p75 = {}, {}
for s in stores:
    m = foods & (sales['store_id'] == s)
    store_all[s] = sales.loc[m, d_cols].sum(axis=0).values.astype(float)
    store_p75[s] = sales.loc[m & (sales['_tier'] == 3), d_cols].sum(axis=0).values.astype(float)

rows = []
for ev in target_events:
    ev_idx = np.array([d_to_idx[d] for d in
                       calendar[calendar['event_name_1'] == ev]['d'] if d in d_to_idx])
    if len(ev_idx) == 0: continue
    for s in stores:
        a, p = store_all[s], store_p75[s]
        base = a[non_ev_idx].mean()
        lift = (a[ev_idx].mean() / base - 1) * 100 if base > 0 else 0
        p75_norm = p[non_ev_idx].sum() / (a[non_ev_idx].sum() + 1e-8)
        p75_ev = p[ev_idx].sum() / (a[ev_idx].sum() + 1e-8)
        rows.append({'event': ev, 'store': s, 'lift': lift,
                     'p75_shift': (p75_ev - p75_norm) * 100})
df_lift = pd.DataFrame(rows)

# === 4-panel visualization ===
fig, axes = plt.subplots(2, 2, figsize=(22, 16))

# 1: Total lift heatmap
pv = df_lift.pivot(index='event', columns='store', values='lift')
sns.heatmap(pv, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            ax=axes[0,0], linewidths=0.5)
axes[0,0].set_title('FOODS Total Lift % by Event × Store', fontsize=12)

# 2: P75 share shift heatmap
pv2 = df_lift.pivot(index='event', columns='store', values='p75_shift')
sns.heatmap(pv2, annot=True, fmt='.2f', cmap='RdYlBu_r', center=0,
            ax=axes[0,1], linewidths=0.5)
axes[0,1].set_title('P75+ Share Shift (pp) on Event Days — FOODS', fontsize=12)

# 3: Universal vs Regional scatter
es = df_lift.groupby('event').agg(
    mean_lift=('lift','mean'), std_lift=('lift','std')).reset_index()
axes[1,0].scatter(es['mean_lift'], es['std_lift'], s=100, c='steelblue', edgecolors='k')
for _, r in es.iterrows():
    axes[1,0].annotate(r['event'], (r['mean_lift'], r['std_lift']),
                       fontsize=8, xytext=(5,5), textcoords='offset points')
axes[1,0].axhline(es['std_lift'].median(), color='r', ls='--', alpha=.5, label='median std')
axes[1,0].axvline(0, color='gray', ls='--', alpha=.5)
axes[1,0].set_xlabel('Mean Lift %'); axes[1,0].set_ylabel('Std Lift %')
axes[1,0].set_title('Universal (low std) vs Regional (high std)', fontsize=12)
axes[1,0].legend()

# 4: "Indifferent" stores
indf = df_lift[df_lift['lift'].abs() < 2].groupby('store').size().reindex(stores, fill_value=0)
axes[1,1].barh(indf.index, indf.values, color='salmon', edgecolor='k')
axes[1,1].set_xlabel('# events with |lift| < 2%')
axes[1,1].set_title('"Indifferent" Stores — Near-Zero Response Events', fontsize=12)

plt.suptitle('Step 7a: Event Universality vs Regional Specificity', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR + '26_event_universality.png', dpi=150, bbox_inches='tight')
plt.show()

# === Summary ===
med = es['std_lift'].median()
print("=== Universal Events (std < median) ===")
print(es[es['std_lift'] < med].sort_values('mean_lift', ascending=False)[
    ['event','mean_lift','std_lift']].to_string(index=False))
print("\n=== Regional Events (std >= median) ===")
print(es[es['std_lift'] >= med].sort_values('std_lift', ascending=False)[
    ['event','mean_lift','std_lift']].to_string(index=False))
print("\n=== 空振りイベント per store (|lift| < 2%) ===")
for s in stores:
    blanks = df_lift[(df_lift['store']==s) & (df_lift['lift'].abs() < 2)]['event'].tolist()
    if blanks: print(f"  {s}: {', '.join(blanks)}")

In [ ]:
# --- Step 7b: ラマダン期間の深層リフト分析 ---

# === ラマダン期間の特定 ===
ram_starts = calendar[calendar['event_name_1'] == 'Ramadan starts'][['d','date','year']].copy()
eid_dates  = calendar[calendar['event_name_1'] == 'Eid al-Fitr'][['d','date','year']].copy()
ram_starts['d_num'] = ram_starts['d'].str[2:].astype(int)
eid_dates['d_num']  = eid_dates['d'].str[2:].astype(int)

ram_periods = []
for _, rs in ram_starts.iterrows():
    yr = rs['year']
    eid = eid_dates[eid_dates['year'] == yr]
    end_d = (eid.iloc[0]['d_num'] + 3) if len(eid) > 0 else (rs['d_num'] + 32)
    end_d = min(end_d, len(d_cols))
    base_start = max(1, rs['d_num'] - 30)
    ram_periods.append({
        'year': yr, 'ram_start': rs['d_num'], 'ram_end': end_d,
        'base_start': base_start, 'base_end': rs['d_num'] - 1,
    })
    print(f"  {yr}: Ramadan d_{rs['d_num']}~d_{end_d}, baseline d_{base_start}~d_{rs['d_num']-1}")

def d_range(start, end):
    return [i for i in range(start - 1, min(end, len(d_cols)))]

# === 店舗 × dept × tier 日次売上の事前計算 ===
depts = sorted(sales['dept_id'].unique())
grp_daily = {}
for (s, dept, tier), g in sales.groupby(['store_id', 'dept_id', '_tier']):
    grp_daily[(s, dept, tier)] = g[d_cols].sum(axis=0).values.astype(float)
print(f"  Precomputed {len(grp_daily)} groups (store×dept×tier)")

# === ラマダン・リフト計算 ===
tier_labels = {0: '<P25', 1: 'P25-50', 2: 'P50-75', 3: 'P75+'}
lift_rows = []
for period in ram_periods:
    ram_idx  = d_range(period['ram_start'], period['ram_end'])
    base_idx = d_range(period['base_start'], period['base_end'])
    if not ram_idx or not base_idx: continue
    for s in stores:
        for dept in depts:
            for tier in range(4):
                key = (s, dept, tier)
                if key not in grp_daily: continue
                d = grp_daily[key]
                base_avg = d[base_idx].mean()
                ram_avg  = d[ram_idx].mean()
                lift = (ram_avg / base_avg - 1) * 100 if base_avg > 1 else 0
                lift_rows.append({'year': period['year'], 'store': s,
                                  'dept': dept, 'tier': tier, 'lift': lift})
df_ram = pd.DataFrame(lift_rows)
del grp_daily

# === ramadan_sensitive スコア ===
# FOODS P75+ と HOUSEHOLD P75+ の正リフトを重視 (単なる季節需要と区別)
score_rows = []
for s in stores:
    sd = df_ram[df_ram['store'] == s]
    foods_p75  = sd[(sd['dept'].str.startswith('FOODS'))   & (sd['tier'] == 3)]['lift'].mean()
    hh_p75     = sd[(sd['dept'].str.startswith('HOUSEHOLD'))& (sd['tier'] == 3)]['lift'].mean()
    foods_all  = sd[sd['dept'].str.startswith('FOODS')]['lift'].mean()
    overall    = sd['lift'].mean()
    # スコア: 高価格帯の cross-category リフト
    score = max(0, foods_p75) * 0.4 + max(0, hh_p75) * 0.3 + max(0, foods_all) * 0.3
    score_rows.append({'store': s, 'foods_p75': foods_p75, 'hh_p75': hh_p75,
                       'foods_all': foods_all, 'overall': overall, 'ramadan_sensitive': score})
df_score = pd.DataFrame(score_rows).set_index('store').sort_values('ramadan_sensitive', ascending=False)
print("\n=== Ramadan Sensitivity Score ===")
print(df_score.round(2).to_string())

# === 4-panel visualization ===
fig, axes = plt.subplots(2, 2, figsize=(22, 16))

# Panel 1: Store × Dept heatmap (全tier平均)
avg_sd = df_ram.groupby(['store','dept'])['lift'].mean().reset_index()
pv1 = avg_sd.pivot(index='dept', columns='store', values='lift')
sns.heatmap(pv1, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            ax=axes[0,0], linewidths=0.5)
axes[0,0].set_title('Ramadan Avg Lift % — Store × Dept (all tiers)', fontsize=12)

# Panel 2: Store × Tier heatmap (FOODS のみ)
foods_ram = df_ram[df_ram['dept'].str.startswith('FOODS')]
avg_st = foods_ram.groupby(['store','tier'])['lift'].mean().reset_index()
avg_st['tier_label'] = avg_st['tier'].map(tier_labels)
pv2 = avg_st.pivot(index='store', columns='tier_label', values='lift')
pv2 = pv2[['<P25','P25-50','P50-75','P75+']]  # order columns
sns.heatmap(pv2, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            ax=axes[0,1], linewidths=0.5)
axes[0,1].set_title('FOODS Ramadan Lift % by Price Tier × Store', fontsize=12)

# Panel 3: Eid al-Fitr spike (top 3 sensitive stores, averaged across years)
top3 = df_score.head(3).index.tolist()
ax3 = axes[1,0]
for s in top3:
    m = (sales['store_id'] == s) & (sales['cat_id'] == 'FOODS')
    daily = sales.loc[m, d_cols].sum(axis=0).values
    avg_window = np.zeros(15)
    n_years = 0
    for _, ed in eid_dates.iterrows():
        dn = ed['d_num']
        for offset in range(-7, 8):
            idx = dn - 1 + offset
            if 0 <= idx < len(daily):
                avg_window[offset + 7] += daily[idx]
        n_years += 1
    avg_window /= max(n_years, 1)
    ax3.plot(range(-7, 8), avg_window, marker='o', ms=4, label=s)
ax3.axvline(0, color='red', ls='--', alpha=0.7, label='Eid al-Fitr')
ax3.set_xlabel('Days from Eid al-Fitr')
ax3.set_ylabel('Avg Daily FOODS Sales')
ax3.set_title('Eid al-Fitr Spike — FOODS (avg across years)', fontsize=12)
ax3.legend(fontsize=9)

# Panel 4: ramadan_sensitive ranking
ax4 = axes[1,1]
colors = ['teal' if v > 5 else 'gray' for v in df_score['ramadan_sensitive']]
ax4.barh(df_score.index, df_score['ramadan_sensitive'], color=colors, edgecolor='k')
ax4.set_xlabel('Ramadan Sensitive Score')
ax4.set_title('Ramadan Sensitivity Ranking', fontsize=12)
ax4.axvline(5, color='red', ls='--', alpha=0.5, label='threshold=5')
ax4.legend()

plt.suptitle('Step 7b: Ramadan Deep Lift Analysis', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR + '27_ramadan_deep_lift.png', dpi=150, bbox_inches='tight')
plt.show()

# === FOODS_2 & HOUSEHOLD 高価格帯の有意リフト確認 ===
print("\n=== FOODS_2 / FOODS_3 / HOUSEHOLD — P75+ Ramadan Lift (store avg) ===")
for dept_check in ['FOODS_2', 'FOODS_3', 'HOUSEHOLD_1', 'HOUSEHOLD_2']:
    sub = df_ram[(df_ram['dept'] == dept_check) & (df_ram['tier'] == 3)]
    if len(sub) == 0: continue
    store_avg = sub.groupby('store')['lift'].mean().sort_values(ascending=False)
    sig = store_avg[store_avg > 5]
    print(f"\n  {dept_check} P75+: {len(sig)} stores with >5% lift")
    for s, v in sig.items():
        print(f"    {s}: +{v:.1f}%")

# Cleanup
sales.drop(columns=['_tier'], inplace=True, errors='ignore')

In [ ]:
# --- Step 7c: 価格帯行動プロファイリング & 既存特徴量の冗長性検証 ---

# === 1. dept別 Z-score → プレミアム品定義 (z > 2.0) ===
# item_avg, dept_q は Step 7a で計算済み
# dept別の平均・標準偏差を算出
dept_stats = {}
dept_items_list = defaultdict(list)
for item, price in item_avg.items():
    dept = '_'.join(item.split('_')[:2])
    dept_items_list[dept].append((item, price))
for dept, items in dept_items_list.items():
    prices = np.array([p for _, p in items])
    dept_stats[dept] = {'mean': prices.mean(), 'std': prices.std()}
del dept_items_list

# item → is_premium (z > 2.0)
item_premium_z = {}
for item, price in item_avg.items():
    dept = '_'.join(item.split('_')[:2])
    st = dept_stats[dept]
    z = (price - st['mean']) / st['std'] if st['std'] > 0 else 0
    item_premium_z[item] = int(z > 2.0)

n_prem = sum(item_premium_z.values())
print(f"=== プレミアム品 (Z > 2.0): {n_prem} / {len(item_premium_z)} items ({n_prem/len(item_premium_z)*100:.1f}%) ===")
for dept, st in sorted(dept_stats.items()):
    n = sum(1 for item, v in item_premium_z.items() if v == 1 and '_'.join(item.split('_')[:2]) == dept)
    print(f"  {dept}: mean=${st['mean']:.2f}, std=${st['std']:.2f}, premium={n}")

# === 2. store × dept の premium_share 算出 ===
sales['_zprem'] = sales['item_id'].map(item_premium_z).fillna(0).astype('int8')
premium_share = {}
for (s, dept), g in sales.groupby(['store_id', 'dept_id']):
    total = g[d_cols].sum().sum()
    prem = g[g['_zprem'] == 1][d_cols].sum().sum()
    premium_share[(s, dept)] = prem / total * 100 if total > 0 else 0

df_ps = pd.DataFrame([
    {'store': s, 'dept': d, 'premium_share': v}
    for (s, d), v in premium_share.items()
])

# === 3. Luxury Index 算出 (Step 5 から再計算) ===
# anchor items (FOODS_3, 低CV × 高売上 top5)
foods3 = sales[sales['dept_id'] == 'FOODS_3']
item_means = foods3.groupby('item_id')[d_cols].sum().sum(axis=1)
item_stds = foods3.groupby('item_id')[d_cols].std(axis=1).mean(axis=1)
item_cv = item_stds / (item_means + 1e-8)
low_cv = item_cv[item_cv < item_cv.quantile(0.3)]
anchors = item_means.loc[low_cv.index].nlargest(5).index.tolist()

# market_score & luxury_index per store
market_score = {}
luxury_index = {}
for s in stores:
    sm = sales[sales['store_id'] == s]
    market_score[s] = sm[sm['item_id'].isin(anchors)][d_cols].sum().sum()
for s in stores:
    # high-price items (P75+ per dept) sales / market_score
    high_sales = sales[(sales['store_id'] == s) & (sales['_tier'] == 3)][d_cols].sum().sum()
    luxury_index[s] = high_sales / market_score[s] if market_score[s] > 0 else 0

# === 4. Visualization (4 panels) ===
fig, axes = plt.subplots(2, 2, figsize=(22, 16))

# Panel 1: store × dept premium_share heatmap
pv = df_ps.pivot(index='dept', columns='store', values='premium_share')
sns.heatmap(pv, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[0,0], linewidths=0.5)
axes[0,0].set_title('store_dept_premium_share (%) — Z>2.0 items', fontsize=12)

# Panel 2: premium_share vs luxury_index (scatter, store-level avg)
store_avg_ps = df_ps.groupby('store')['premium_share'].mean()
lux_vals = pd.Series(luxury_index)
ax2 = axes[0,1]
ax2.scatter(lux_vals, store_avg_ps, s=120, c='darkorange', edgecolors='k')
for s in stores:
    ax2.annotate(s, (lux_vals[s], store_avg_ps[s]), fontsize=9, xytext=(5,5),
                 textcoords='offset points')
from scipy import stats as sp_stats
r, p = sp_stats.pearsonr(lux_vals[stores], store_avg_ps[stores])
ax2.set_xlabel('Luxury Index'); ax2.set_ylabel('Avg Premium Share %')
ax2.set_title(f'Premium Share vs Luxury Index (r={r:.3f}, p={p:.3f})', fontsize=12)

# Panel 3: premium_share vs pb_ratio redundancy check
# 既存 pb_ratio (preprocess.py Phase 1.5 で算出) を再計算
pb_ratio_store = {}
for chunk in pd.read_csv(DATA_DIR + 'sell_prices.csv', chunksize=500_000,
                          usecols=['store_id', 'item_id', 'sell_price']):
    pass  # just to get the structure
# 簡易版: item_avg の P20 以下の売上シェア
dept_p20 = {d: np.percentile(p, 20) for d, p in
            [(d, [item_avg[i] for i in item_avg if '_'.join(i.split('_')[:2]) == d])
             for d in set('_'.join(i.split('_')[:2]) for i in item_avg)]}
item_is_pb = {item: int(item_avg[item] <= dept_p20.get('_'.join(item.split('_')[:2]), 0))
              for item in item_avg}
sales['_pb'] = sales['item_id'].map(item_is_pb).fillna(0).astype('int8')
store_pb = {}
for s in stores:
    sm = sales[sales['store_id'] == s]
    total = sm[d_cols].sum().sum()
    pb = sm[sm['_pb'] == 1][d_cols].sum().sum()
    store_pb[s] = pb / total * 100 if total > 0 else 0

ax3 = axes[1,0]
ax3.scatter(pd.Series(store_pb), store_avg_ps, s=120, c='steelblue', edgecolors='k')
for s in stores:
    ax3.annotate(s, (store_pb[s], store_avg_ps[s]), fontsize=9, xytext=(5,5),
                 textcoords='offset points')
r2, p2 = sp_stats.pearsonr(
    [store_pb[s] for s in stores], [store_avg_ps[s] for s in stores])
ax3.set_xlabel('PB Ratio (P20 share) %'); ax3.set_ylabel('Premium Share %')
ax3.set_title(f'Redundancy Check: Premium Share vs PB Ratio (r={r2:.3f})', fontsize=12)

# Panel 4: premium_share × event lift correlation
# 各店舗の premium_share と主要イベントリフトの相関
ev_lift_store = df_lift.groupby('store')['lift'].mean()
ax4 = axes[1,1]
ax4.scatter(store_avg_ps, ev_lift_store[stores], s=120, c='green', edgecolors='k')
for s in stores:
    ax4.annotate(s, (store_avg_ps[s], ev_lift_store[s]), fontsize=9, xytext=(5,5),
                 textcoords='offset points')
r3, p3 = sp_stats.pearsonr(store_avg_ps[stores].values, ev_lift_store[stores].values)
ax4.set_xlabel('Avg Premium Share %'); ax4.set_ylabel('Avg Event Lift %')
ax4.set_title(f'Premium Share vs Event Lift (r={r3:.3f})', fontsize=12)

plt.suptitle('Step 7c: Price Tier Profiling & Feature Redundancy Check', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR + '28_price_profiling.png', dpi=150, bbox_inches='tight')
plt.show()

# === 5. 冗長性の結論 ===
print(f"\n=== 冗長性チェック ===")
print(f"  premium_share vs luxury_index: r={r:.3f} (p={p:.3f})")
print(f"  premium_share vs pb_ratio:     r={r2:.3f} (p={p2:.3f})")
print(f"  premium_share vs event_lift:   r={r3:.3f} (p={p3:.3f})")
if abs(r2) > 0.8:
    print("  → pb_ratio と高相関 → store_dept_premium_share は pb_ratio を置換可能")
elif abs(r2) < 0.5:
    print("  → pb_ratio と低相関 → store_dept_premium_share は独立した情報を持つ")
else:
    print("  → pb_ratio と中程度の相関 → 両方残す価値あり (要CV検証)")

# === 「所得でないのに高価格品が売れる」異常店舗 ===
print("\n=== 嗜好性シグナル: 所得に反して高価格品が売れる店舗 ===")
lux_med = lux_vals.median()
ps_med = store_avg_ps.median()
for s in stores:
    if lux_vals[s] < lux_med and store_avg_ps[s] > ps_med:
        print(f"  {s}: luxury_index={lux_vals[s]:.3f} (below median) but "
              f"premium_share={store_avg_ps[s]:.2f}% (above median) → 地域嗜好シグナル")

# Cleanup
sales.drop(columns=['_zprem', '_pb', '_tier'], inplace=True, errors='ignore')